# PS-26015 — Watershed Geospatial Pipeline (Colab)

Application of Geospatial Techniques for Watershed Development Outcomes.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

This notebook pulls Sentinel-2 imagery + ESA WorldCover labels for the current AOI (Kadwanchi watershed, Jalna dist., Maharashtra — a documented Indo-German Watershed Development Programme site with real check dams/percolation tank, chosen after an earlier placeholder AOI turned out to have 0% water pixels in ground truth), builds a 6-channel (R,G,B,NIR,NDVI,NDWI) LULC training set, trains:
- **Model 1**: U-Net LULC segmentation (7 classes)
- **Model 2**: Siamese change-detection U-Net (5 classes), with a **Tier-1 rule-based fallback** that needs no training at all

...then applies a rule-based recommendation engine and renders visualization outputs (static figure + interactive Folium map).

All data/checkpoints/outputs are written under your Google Drive so they survive Colab session resets. **Swap `AOI_BBOX` in the Config cell for your real target watershed** as soon as it's decided — everything downstream is AOI-agnostic.


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — go to Runtime > Change runtime type > T4 GPU, then Runtime > Restart and run all.')


## Install packages not preinstalled in Colab


In [ ]:
!pip install -q segmentation-models-pytorch pystac-client rasterio geopandas folium


## Mount Google Drive (for persistence across session resets)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/watershed_ps26015')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_LABELS = BASE_DIR / 'data' / 'labels'
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
OUTPUTS_DIR = BASE_DIR / 'outputs'
# Tiles are LOCAL Colab disk, not Drive: writing hundreds-to-thousands of small compressed
# .npz files (base patches x 8 augmentations) to Drive is extremely slow (Drive's sync layer
# adds real per-file overhead), and tiles are cheaply regenerable from the persisted
# stack6/mask rasters above -- no reason to pay that cost or use Drive space for them.
TILES_DIR = Path('/content/tiles')
for d in (DATA_RAW, DATA_LABELS, DATA_PROCESSED, TILES_DIR, MODELS_DIR, OUTPUTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Working under:', BASE_DIR, ' (tiles under', TILES_DIR, ', local disk, not synced to Drive)')


## Config

**AOI: Kadwanchi watershed, Jalna dist., Maharashtra.** Recompute `WORLDCOVER_TILE` (3x3-degree grid, lower-left-corner naming, e.g. `N18E075`) if you change `AOI_BBOX` again.


In [ ]:
AOI_NAME = 'kadwanchi_watershed'
AOI_CENTER_LAT = 19.8830
AOI_CENTER_LON = 75.9910
AOI_BBOX = (75.9508, 19.8420, 76.0312, 19.9240)  # (minx, miny, maxx, maxy) EPSG:4326, ~9km x 9km -- enlarged from 6km after a live run showed 0% water predictions; verified this captures a real nearby reservoir (water 0.34% -> 2.79%)

WORLDCOVER_TILE = 'N18E075'
def worldcover_url_for_tile(tile):
    return ('https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/'
            f'ESA_WorldCover_10m_2021_v200_{tile}_Map.tif')
WORLDCOVER_URL = worldcover_url_for_tile(WORLDCOVER_TILE)

# Auxiliary training-only AOIs (single recent date, no change-detection pair) --
# Kadwanchi alone is ~97% agriculture+sparse-vegetation, so dense vegetation,
# barren/degraded land, and rivers/large water bodies had little-to-no training
# signal. Each verified via a ground-truth class-distribution check before adding
# (ESA WorldCover clip + histogram):
#   - Tamhini Ghat, Western Ghats, Pune dist., Maharashtra: 63.1% tree cover
#   - Donimalai iron-ore mine, Sandur, Ballari dist., Karnataka: 4.5% bare/sparse
#     vegetation (mining exposes ground unambiguously; plain 'degraded' farmland
#     did not register as bare in this global dataset -- Anantapur city and the
#     Chambal ravine belt were tried first and both failed this check)
#   - Jayakwadi Dam / Godavari river, Paithan, Aurangabad dist., Maharashtra:
#     49.0% water (one of Maharashtra's largest reservoirs) -- added after a
#     live 'unseen location' query (a river through a dense industrial city)
#     showed the model failing to trace a real river. Deliberately a real dam
#     site, not a city -- rivers are core watershed infrastructure and in
#     scope; general city coverage is not (see documentation.md).
AUX_AOIS = [
    {'name': 'tamhini_ghat_forest', 'bbox': (73.3800, 18.4088, 73.4654, 18.4898), 'worldcover_tile': 'N18E072'},
    {'name': 'donimalai_barren', 'bbox': (76.5517, 15.0184, 76.6357, 15.0994), 'worldcover_tile': 'N15E075'},
    {'name': 'jayakwadi_dam_water', 'bbox': (75.3270, 19.4453, 75.4130, 19.5263), 'worldcover_tile': 'N18E075'},
]

# Every AOI job the pipeline processes: primary (T1 + T2, for change detection)
# plus the auxiliary training-only sites (single recent date 'S1').
AOI_JOBS = [
    {'name': AOI_NAME, 'bbox': AOI_BBOX, 'worldcover_tile': WORLDCOVER_TILE, 'dates': ['T1', 'T2']},
] + [
    {'name': a['name'], 'bbox': a['bbox'], 'worldcover_tile': a['worldcover_tile'], 'dates': ['S1']}
    for a in AUX_AOIS
]

WORLDCOVER_TO_MYCLASS = {
    10: 1, 20: 3, 30: 3, 40: 2, 50: 5, 60: 4, 70: 4, 80: 0, 90: 0, 95: 1, 100: 3,
}

CLASS_NAMES = {
    0: 'Water body / conservation structure', 1: 'Dense vegetation / forest',
    2: 'Agriculture / cropland', 3: 'Sparse vegetation / grassland',
    4: 'Barren / degraded land', 5: 'Built-up / settlement', 6: 'Fallow / bare agricultural land',
}
NUM_CLASSES = len(CLASS_NAMES)
CLASS_COLORS = {
    0: (66, 135, 245), 1: (34, 102, 51), 2: (154, 205, 50), 3: (189, 183, 107),
    4: (160, 120, 90), 5: (200, 30, 30), 6: (222, 184, 135),
}

# Sentinel for 'no real satellite coverage at this pixel' (a scene whose
# footprint only partially overlapped the AOI -- common near MGRS tile edges).
# Assigned post-argmax in predict_class_map, never a real model output; stays
# outside 0..NUM_CLASSES-1 so NUM_CLASSES (the trained model's output head
# size) is unaffected. See src/config.py for the full incident writeup.
NODATA_CLASS = 255
CLASS_NAMES[NODATA_CLASS] = 'No data / no coverage'
CLASS_COLORS[NODATA_CLASS] = (225, 225, 225)

CHANGE_CLASS_NAMES = {
    0: 'No change', 1: 'New water/conservation structure', 2: 'New construction/built-up',
    3: 'Vegetation/water loss (degradation)', 4: 'Vegetation gain',
}

S2_BANDS = ['red', 'green', 'blue', 'nir']
STAC_API_URL = 'https://earth-search.aws.element84.com/v1'
STAC_COLLECTION = 'sentinel-2-l2a'

PATCH_SIZE = 128
PATCH_OVERLAP = 32
BATCH_SIZE = 16   # Colab T4 has 16GB VRAM -- can go higher than the 4GB-laptop plan's 8-16
NUM_EPOCHS = 25
LR = 1e-4
IN_CHANNELS = 6
FREEZE_ENCODER_EPOCHS = 5

import rasterio
def atomic_raster_write(out_path, data, profile, descriptions=None):
    # Write to a temp path first, then rename into place -- so a Colab disconnect or
    # interrupted cell mid-write can never leave a truncated file sitting at a path the
    # rest of the pipeline trusts as complete (hit for real in this project -- a file
    # that opens fine but fails on the actual pixel read later, confusingly, elsewhere).
    out_path = Path(out_path)
    tmp_path = out_path.with_suffix(out_path.suffix + '.tmp')
    with rasterio.open(tmp_path, 'w', **profile) as dst:
        dst.write(data)
        if descriptions:
            dst.descriptions = descriptions
    tmp_path.replace(out_path)


## 1. Data download
No accounts/registration needed: Sentinel-2 L2A via the public Earth Search STAC API (AWS Open Data COGs), ESA WorldCover 10m labels via public S3. Clips straight to the AOI on read — no full-tile bulk download. Runs for every job in `AOI_JOBS`: the primary AOI (T1 + T2, for change detection) plus the auxiliary training-only AOIs (single date 'S1', added to fix class imbalance -- see the Config cell).


In [ ]:
import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.warp import transform_bounds, reproject, Resampling
from shapely.geometry import box, mapping
from pystac_client import Client

DATE_WINDOWS = {
    'T2': '2024-11-01/2025-03-31', 'T1': '2019-11-01/2020-03-31', 'S1': '2024-11-01/2025-03-31',
}

def _fully_covers(item_bbox, bbox):
    minx, miny, maxx, maxy = bbox
    ib0, ib1, ib2, ib3 = item_bbox
    return ib0 <= minx and ib1 <= miny and ib2 >= maxx and ib3 >= maxy

def search_scene(bbox, date_tag, max_cloud=20, limit=10):
    # Prefers a scene whose own footprint fully covers bbox -- a scene only
    # partially overlapping the AOI is real (confirmed for Kadwanchi's own
    # primary AOI: the plain lowest-cloud pick left 40.7% of the bbox
    # uncovered even though a same-cloud, full-coverage alternative existed
    # in the same search window and was never considered under a
    # cloud-cover-only sort).
    catalog = Client.open(STAC_API_URL)
    search = catalog.search(collections=[STAC_COLLECTION], bbox=bbox, datetime=DATE_WINDOWS[date_tag],
                             query={'eo:cloud_cover': {'lt': max_cloud}}, limit=limit)
    items = list(search.items())
    if not items:
        raise RuntimeError(f'No low-cloud Sentinel-2 scene for bbox={bbox}, date_tag={date_tag} -- widen date range/cloud threshold.')
    full_coverage = [it for it in items if _fully_covers(it.bbox, bbox)]
    pool = full_coverage if full_coverage else items
    if not full_coverage:
        print(f'WARNING: no scene in this window fully covers bbox={bbox} -- picking lowest-cloud partial match.')
    pool.sort(key=lambda it: it.properties.get('eo:cloud_cover', 100))
    return pool[0]

def clip_scene_to_stack(item, bbox, out_path):
    # Always produces an array sized to the FULL requested bbox, regardless
    # of how much of it the matched scene's footprint actually covers -- the
    # previous rio_mask(crop=True) approach silently returned a SMALLER
    # array when the scene didn't fully cover the bbox (an actual shape
    # truncation, not just nodata pixels within a correctly-sized read,
    # which nothing downstream could detect). Reprojecting into a
    # pre-sized destination (same CRS, so this is a resample/pad, not a
    # real reprojection) makes any uncovered area fall out as legitimate
    # zero/nodata pixels instead, which NODATA_CLASS already handles
    # correctly everywhere downstream.
    band_arrays, profile = [], None
    target_transform = target_h = target_w = None
    for band in S2_BANDS:
        href = item.assets[band].href
        with rasterio.open(href) as src:
            if target_transform is None:
                minx, miny, maxx, maxy = transform_bounds('EPSG:4326', src.crs, *bbox)
                res = src.res[0]
                target_w = max(1, round((maxx - minx) / res))
                target_h = max(1, round((maxy - miny) / res))
                target_transform = rasterio.transform.from_origin(minx, maxy, res, res)
                profile = src.profile.copy()
                profile.update(height=target_h, width=target_w, transform=target_transform,
                                count=len(S2_BANDS), dtype='uint16')
            band_data = np.zeros((target_h, target_w), dtype='uint16')
            reproject(source=rasterio.band(src, 1), destination=band_data,
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=target_transform, dst_crs=src.crs,
                      resampling=Resampling.nearest)
            band_arrays.append(band_data)
    stack = np.stack(band_arrays, axis=0)
    atomic_raster_write(out_path, stack, profile, descriptions=tuple(S2_BANDS))
    n_nodata = int(np.all(stack == 0, axis=0).sum())
    print(f'Saved {out_path}  shape={stack.shape}  date={item.datetime.date()}  '
          f"cloud={item.properties.get('eo:cloud_cover'):.1f}%  nodata={n_nodata}px")

def download_worldcover(bbox, worldcover_tile, out_path):
    vsi_url = f'/vsicurl/{worldcover_url_for_tile(worldcover_tile)}'
    with rasterio.open(vsi_url) as src:
        geom = [mapping(box(*bbox))]
        data, transform = rio_mask(src, geom, crop=True)
        profile = src.profile.copy()
        profile.update(height=data.shape[1], width=data.shape[2], transform=transform)
    atomic_raster_write(out_path, data, profile)
    print(f'Saved {out_path}  shape={data.shape}')

def run_download_job(job):
    name, bbox, tile, dates = job['name'], job['bbox'], job['worldcover_tile'], job['dates']
    print(f'\n=== AOI: {name}  bbox={bbox}  dates={dates} ===')
    for date_tag in dates:
        item = search_scene(bbox, date_tag)
        print(f'{date_tag}: {item.id}  ({item.datetime.date()}, cloud={item.properties.get("eo:cloud_cover"):.1f}%)')
        clip_scene_to_stack(item, bbox, DATA_RAW / f'{name}_{date_tag}_rgbnir.tif')
    download_worldcover(bbox, tile, DATA_LABELS / f'{name}_worldcover.tif')


In [ ]:
for job in AOI_JOBS:
    run_download_job(job)


## 2. Preprocessing
6-channel stack (R,G,B,NIR,NDVI,NDWI) + WorldCover labels reprojected onto the imagery grid and remapped to our 7-class scheme.


In [ ]:
from rasterio.warp import reproject, Resampling
EPS = 1e-6

def compute_indices(stack):
    red, green, blue, nir = stack.astype('float32')
    ndvi = (nir - red) / (nir + red + EPS)
    ndwi = (green - nir) / (green + nir + EPS)
    return ndvi, ndwi

def build_6channel_stack(raw_path, out_path):
    with rasterio.open(raw_path) as src:
        stack = src.read()
        profile = src.profile.copy()
    ndvi, ndwi = compute_indices(stack)
    rgb_nir = stack.astype('float32') / 10000.0
    six = np.concatenate([rgb_nir, ndvi[None], ndwi[None]], axis=0)
    profile.update(count=6, dtype='float32')
    atomic_raster_write(out_path, six, profile, descriptions=('red', 'green', 'blue', 'nir', 'ndvi', 'ndwi'))
    print(f'Saved {out_path}  shape={six.shape}')
    return profile

def rasterize_labels(worldcover_path, target_profile, out_path):
    with rasterio.open(worldcover_path) as wc_src:
        wc_data = wc_src.read(1)
        wc_crs, wc_transform = wc_src.crs, wc_src.transform
    dst_h, dst_w = target_profile['height'], target_profile['width']
    aligned = np.zeros((dst_h, dst_w), dtype='uint8')
    reproject(source=wc_data, destination=aligned, src_transform=wc_transform, src_crs=wc_crs,
              dst_transform=target_profile['transform'], dst_crs=target_profile['crs'],
              dst_resolution=(target_profile['transform'].a, -target_profile['transform'].e),
              resampling=Resampling.nearest)
    remapped = np.full_like(aligned, fill_value=6)
    for wc_code, my_class in WORLDCOVER_TO_MYCLASS.items():
        remapped[aligned == wc_code] = my_class
    mask_profile = target_profile.copy()
    mask_profile.update(count=1, dtype='uint8', nodata=None)
    atomic_raster_write(out_path, remapped[None], mask_profile)
    print(f'Saved {out_path}  shape={remapped.shape}  '
          f'class counts={dict(zip(*np.unique(remapped, return_counts=True)))}')

# An NDVI-based label refinement (splitting WorldCover's cropland/shrub-grassland classes
# using vegetation vigor, calibrated against real Bhuvan AOI-wise LULC statistics for
# Kadwanchi) was tried here and reverted: even scoped to just its calibration site, it
# underperformed plain WorldCover labels once retrained (65.9% mean IoU baseline vs 61.8%
# all-sites / 63.0% scoped -- barren/fallow IoU stayed below baseline in every variant). A
# per-pixel NDVI threshold has no spatial coherence, so matching the real aggregate
# proportion didn't translate into learnable, clean class boundaries. See documentation.md
# section 6a for the full history. The real fix is a real Bhuvan shapefile, not a heuristic.


In [ ]:
profiles = {}  # keyed by (aoi_name, date_tag) -- Model 1 training only needs the primary AOI's,
                # but we keep them all in case you want to inspect an aux AOI's grid later.
for job in AOI_JOBS:
    name = job['name']
    worldcover_path = DATA_LABELS / f'{name}_worldcover.tif'
    for date_tag in job['dates']:
        raw_path = DATA_RAW / f'{name}_{date_tag}_rgbnir.tif'
        stack_out = DATA_PROCESSED / f'{name}_{date_tag}_stack6.tif'
        mask_out = DATA_PROCESSED / f'{name}_{date_tag}_mask.tif'
        profiles[(name, date_tag)] = build_6channel_stack(raw_path, stack_out)
        rasterize_labels(worldcover_path, profiles[(name, date_tag)], mask_out)


## 3. Tiling
128x128 patches with overlap, 8x dihedral augmentation, dropping mostly-nodata patches, train/val split written to `manifest.csv`.


In [ ]:
import csv
STRIDE = PATCH_SIZE - PATCH_OVERLAP
VAL_FRACTION = 0.15
MAX_NODATA_FRACTION = 0.05

def augmentations(img, mask):
    out = []
    for k in range(4):
        img_r = np.rot90(img, k, axes=(1, 2))
        mask_r = np.rot90(mask, k)
        out.append((img_r, mask_r))
        out.append((np.flip(img_r, axis=2), np.flip(mask_r, axis=1)))
    return out

def tile_pair(aoi_name, date_tag):
    # Returns base-patch records (not individual augmented files) -- the
    # train/val split below is decided per base patch, before augmentation,
    # so a patch and its 8 rotated/flipped copies always land in the same
    # split (splitting augmented files independently let val silently
    # contain rotated copies of train pixels -- not real generalization).
    stack_path = DATA_PROCESSED / f'{aoi_name}_{date_tag}_stack6.tif'
    mask_path = DATA_PROCESSED / f'{aoi_name}_{date_tag}_mask.tif'
    with rasterio.open(stack_path) as s:
        img = s.read()
    with rasterio.open(mask_path) as m:
        msk = m.read(1)
    C, H, W = img.shape
    patch_records, patch_id = [], 0
    for y in range(0, H - PATCH_SIZE + 1, STRIDE):
        for x in range(0, W - PATCH_SIZE + 1, STRIDE):
            img_p = img[:, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            msk_p = msk[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            nodata_frac = float(np.mean(np.all(img_p == 0, axis=0)))
            if nodata_frac > MAX_NODATA_FRACTION:
                continue
            files = []
            for aug_idx, (img_a, msk_a) in enumerate(augmentations(img_p, msk_p)):
                fname = f'{aoi_name}_{date_tag}_p{patch_id:04d}_a{aug_idx}.npz'
                np.savez_compressed(TILES_DIR / fname, image=img_a.astype('float32'), mask=msk_a.astype('uint8'))
                files.append(fname)
            patch_records.append({'y': y, 'x': x, 'files': files})
            patch_id += 1
    n_tiles = sum(len(r['files']) for r in patch_records)
    print(f'{aoi_name} {date_tag}: {patch_id} base patches -> {n_tiles} tiles')
    return patch_records

# Split per AOI+date via a spatial block (highest-y rows of that scene's patch
# grid held out as val), not a random shuffle across the pooled patch list --
# random shuffling put spatially adjacent (overlapping, since STRIDE < PATCH_SIZE)
# patches on both sides of the split, which are highly correlated and leak the
# same way augmented duplicates did. Doing this per AOI+date (not one global
# block) keeps every AOI represented in both train and val.
all_files = []
for job in AOI_JOBS:
    for date_tag in job['dates']:
        patch_records = tile_pair(job['name'], date_tag)
        if not patch_records:
            continue
        patch_records.sort(key=lambda r: (r['y'], r['x']))
        n_val_patches = max(1, int(len(patch_records) * VAL_FRACTION))
        val_patches = patch_records[-n_val_patches:] if len(patch_records) > 1 else []
        val_ys = {r['y'] for r in val_patches}
        for r in patch_records:
            split = 'val' if r['y'] in val_ys else 'train'
            for fname in r['files']:
                all_files.append((fname, split))

manifest_path = TILES_DIR / 'manifest.csv'
with open(manifest_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filename', 'split'])
    for fname, split in all_files:
        writer.writerow([fname, split])
n_val = sum(1 for _, split in all_files if split == 'val')
print(f'Total tiles: {len(all_files)}  (train={len(all_files)-n_val}, val={n_val})')
if len(all_files) < 200:
    print('NOTE: small tile count for this placeholder AOI -- a real (larger) watershed AOI, '
          'or more scene dates, will give far more training tiles.')


## 4. PyTorch Dataset


In [ ]:
from torch.utils.data import Dataset, DataLoader

class WatershedTileDataset(Dataset):
    def __init__(self, split='train', manifest_path=None):
        manifest_path = manifest_path or (TILES_DIR / 'manifest.csv')
        with open(manifest_path) as f:
            rows = list(csv.DictReader(f))
        self.files = [r['filename'] for r in rows if r['split'] == split]
        if not self.files:
            raise RuntimeError(f"No '{split}' tiles found -- run the data/preprocessing/tiling cells first.")
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        npz = np.load(TILES_DIR / self.files[idx])
        image = torch.from_numpy(npz['image'])
        mask = torch.from_numpy(npz['mask']).long()
        return image, mask


In [ ]:
def confusion_matrix_from_arrays(y_true, y_pred, num_classes):
    y_true, y_pred = y_true.ravel(), y_pred.ravel()
    idx = y_true * num_classes + y_pred
    counts = np.bincount(idx, minlength=num_classes * num_classes)
    return counts.reshape(num_classes, num_classes).astype('int64')

def metrics_from_confusion(cm, class_names):
    num_classes = cm.shape[0]
    support = cm.sum(axis=1)
    per_class = {}
    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        name = class_names.get(c, str(c))
        if support[c] == 0:
            per_class[name] = {'support': 0, 'precision': None, 'recall': None, 'iou': None, 'f1': None}
            continue
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        per_class[name] = {'support': int(support[c]), 'precision': round(float(precision), 3),
                            'recall': round(float(recall), 3), 'iou': round(float(iou), 3), 'f1': round(float(f1), 3)}
    pixel_accuracy = float(np.trace(cm)) / float(cm.sum()) if cm.sum() > 0 else 0.0
    present_ious = [v['iou'] for v in per_class.values() if v['iou'] is not None]
    mean_iou = float(np.mean(present_ious)) if present_ious else 0.0
    return {'per_class': per_class, 'pixel_accuracy': round(pixel_accuracy, 4), 'mean_iou': round(mean_iou, 4),
            'classes_absent_from_val_set': [n for n, v in per_class.items() if v['support'] == 0]}

def print_metrics_report(metrics):
    print(f"Overall pixel accuracy: {metrics['pixel_accuracy']*100:.1f}%")
    print(f"Mean IoU (over classes present in val set): {metrics['mean_iou']*100:.1f}%")
    if metrics['classes_absent_from_val_set']:
        print('NOTE: absent from val set (no reference labels to check against):', ', '.join(metrics['classes_absent_from_val_set']))
    print(f"\n{'Class':<38}{'Support':>9}{'Precision':>11}{'Recall':>9}{'IoU':>7}{'F1':>7}")
    for name, v in metrics['per_class'].items():
        if v['support'] == 0:
            print(f"{name:<38}{'0':>9}{'  -- no reference labels in val set --':>28}")
        else:
            print(f"{name:<38}{v['support']:>9}{v['precision']:>11.3f}{v['recall']:>9.3f}{v['iou']:>7.3f}{v['f1']:>7.3f}")

def plot_confusion_matrix(cm, class_names, out_path):
    import matplotlib.pyplot as plt  # local import: this cell runs before the Demo section's plt import
    n = cm.shape[0]
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype='float64'), where=row_sums != 0)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    labels = [class_names.get(i, str(i)) for i in range(n)]
    ax.set_xticks(range(n)); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Model 1 confusion matrix (row-normalized)')
    for i in range(n):
        for j in range(n):
            if cm[i, j] > 0:
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        fontsize=7, color='white' if cm_norm[i, j] > 0.5 else 'black')
    fig.colorbar(im, ax=ax, label='fraction of true class')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f'Saved confusion matrix figure: {out_path}')


## 5. Model 1 — LULC U-Net
ResNet18 encoder (ImageNet weights), 6-channel input, 7-class output. Frozen encoder for the first few epochs, then fine-tuned end-to-end.


In [ ]:
import time
import segmentation_models_pytorch as smp

def build_model1():
    return smp.Unet(encoder_name='resnet18', encoder_weights='imagenet',
                     in_channels=IN_CHANNELS, classes=NUM_CLASSES)

def set_encoder_trainable(model, trainable: bool):
    for p in model.encoder.parameters():
        p.requires_grad = trainable

def compute_class_weights(dataset, num_classes):
    # Median-frequency balancing (Eigen & Fergus; SegNet). A real, hit result
    # motivates this: a plain unweighted loss, picking the checkpoint by lowest
    # aggregate val_loss, produced a checkpoint where Fallow (~0.25-0.6% of
    # pixels) had precision/recall/IoU all exactly 0.000 -- a class that rare
    # barely moves an aggregate loss either way, so there was no real gradient
    # pressure to ever learn it. Plain inverse-frequency (1/freq) would swing
    # too far the other way (~150x weight on the rarest class, destabilizing
    # training); median-frequency balancing is the standard, better-behaved fix.
    counts = np.zeros(num_classes, dtype='int64')
    for i in range(len(dataset)):
        _, mask = dataset[i]
        counts += np.bincount(mask.numpy().ravel(), minlength=num_classes)
    freq = counts / counts.sum()
    present = freq > 0
    median_freq = np.median(freq[present])
    weights = np.where(present, median_freq / np.maximum(freq, 1e-12), 0.0)
    print('Class weights (median-frequency balanced):')
    for c in range(num_classes):
        print(f'  {CLASS_NAMES.get(c, c):<38} count={counts[c]:>10}  freq={freq[c]*100:6.2f}%  weight={weights[c]:6.3f}')
    return torch.tensor(weights, dtype=torch.float32)

def run_epoch(model, loader, loss_fn, optimizer, scaler, device, train, num_classes=None):
    # num_classes given (only meaningful when train=False) -> also accumulate a
    # confusion matrix over the val set in the SAME pass already used for
    # val_loss, reusing the logits already computed rather than a second forward
    # pass -- this is what lets checkpoint selection use mean IoU below at no
    # extra compute cost.
    model.train(train)
    total_loss, n_batches = 0.0, 0
    cm = np.zeros((num_classes, num_classes), dtype='int64') if num_classes else None
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=device.type, enabled=(device.type=='cuda')):
                logits = model(images)
                loss = loss_fn(logits, masks)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            elif cm is not None:
                preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
                cm += confusion_matrix_from_arrays(masks.cpu().numpy(), preds, num_classes)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1), cm


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

train_ds = WatershedTileDataset('train')
val_ds = WatershedTileDataset('val')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train tiles: {len(train_ds)}  Val tiles: {len(val_ds)}')

model1 = build_model1().to(device)
set_encoder_trainable(model1, False)

class_weights = compute_class_weights(train_ds, NUM_CLASSES).to(device)
dice = smp.losses.DiceLoss(mode='multiclass')
ce = torch.nn.CrossEntropyLoss(weight=class_weights)
loss_fn = lambda logits, target: dice(logits, target) + ce(logits, target)
optimizer = torch.optim.Adam(model1.parameters(), lr=LR)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type=='cuda'))

# Checkpoint selection is by mean IoU, not aggregate val_loss -- a rare class
# barely moves an aggregate loss either way, so val_loss-based selection can
# happily save a checkpoint where that class has completely collapsed (a real
# result this project hit: Fallow IoU 0.000). Computed from the confusion
# matrix run_epoch already accumulates during the val pass, no extra cost.
best_mean_iou = -1.0
ckpt_path = MODELS_DIR / 'model1_lulc_unet.pt'

for epoch in range(1, NUM_EPOCHS + 1):
    if epoch == FREEZE_ENCODER_EPOCHS + 1:
        print('Unfreezing encoder.')
        set_encoder_trainable(model1, True)
    t0 = time.time()
    train_loss, _ = run_epoch(model1, train_loader, loss_fn, optimizer, scaler, device, train=True)
    val_loss, val_cm = run_epoch(model1, val_loader, loss_fn, optimizer, scaler, device, train=False, num_classes=NUM_CLASSES)
    mean_iou = metrics_from_confusion(val_cm, CLASS_NAMES)['mean_iou']
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_mean_iou={mean_iou:.4f}  ({time.time()-t0:.1f}s)')
    if mean_iou > best_mean_iou:
        best_mean_iou = mean_iou
        torch.save({'model_state': model1.state_dict(), 'epoch': epoch, 'val_loss': val_loss, 'val_mean_iou': mean_iou}, ckpt_path)
        print(f'  -> saved best checkpoint ({ckpt_path.name}, mean_iou={mean_iou:.4f})')
print(f'Done. Best val_mean_iou={best_mean_iou:.4f}. Checkpoint: {ckpt_path}')


## 5b. Model 1 accuracy report
Per-class precision/recall/IoU/F1 + overall pixel accuracy + mean IoU on the held-out val tiles, plus a confusion-matrix figure -- the numbers to actually defend to judges, instead of eyeballing the LULC map. Classes absent from the val set are reported as 'no reference labels' rather than a misleading 0.0 (small AOIs may not have every class in val). confusion_matrix_from_arrays/metrics_from_confusion/etc. were defined earlier (Model 1 training needs them per-epoch, for IoU-based checkpoint selection) -- this cell just runs the final report against the saved best checkpoint.


In [ ]:
model1.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state'])
model1.eval()
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype='int64')
with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, enabled=(device.type=='cuda')):
            logits = model1(images)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        cm += confusion_matrix_from_arrays(masks.numpy(), preds, NUM_CLASSES)

metrics = metrics_from_confusion(cm, CLASS_NAMES)
print_metrics_report(metrics)
plot_confusion_matrix(cm, CLASS_NAMES, OUTPUTS_DIR / 'model1_confusion_matrix.png')


## 6. Tier-1 rule-based change detection (guaranteed fallback, no training)
Diffs two Model-1 class maps directly. This is the demo-safe path if Model 2 (next section) doesn't finish training in time.


In [ ]:
from scipy import ndimage

WATER, DENSE_VEG, AGRI, SPARSE_VEG, BARREN, BUILTUP, FALLOW = range(7)
MIN_BLOB_PIXELS_DEFAULT = 12  # ~100 sq m at 10m/px

def diff_to_change_map(class_t1, class_t2):
    change = np.zeros(class_t1.shape, dtype='uint8')
    change[np.isin(class_t1, [BARREN, SPARSE_VEG, AGRI, FALLOW]) & (class_t2 == WATER)] = 1
    change[np.isin(class_t1, [BARREN, SPARSE_VEG, AGRI, FALLOW, WATER]) & (class_t2 == BUILTUP)] = 2
    change[np.isin(class_t1, [DENSE_VEG, WATER]) & (class_t2 == BARREN)] = 3
    change[np.isin(class_t1, [BARREN, FALLOW, SPARSE_VEG]) & np.isin(class_t2, [DENSE_VEG, AGRI])] = 4
    return change

def filter_small_blobs(change_map, min_pixels=MIN_BLOB_PIXELS_DEFAULT):
    cleaned = change_map.copy()
    for cls in np.unique(change_map):
        if cls == 0:
            continue
        cls_mask = change_map == cls
        labeled, n = ndimage.label(cls_mask)
        sizes = ndimage.sum(cls_mask, labeled, range(1, n + 1))
        for blob_id, size in enumerate(sizes, start=1):
            if size < min_pixels:
                cleaned[labeled == blob_id] = 0
    return cleaned

def geofence_mask(change_map, watershed_mask):
    if watershed_mask is None:
        return change_map
    out = change_map.copy()
    out[~watershed_mask] = 0
    return out

def run_tier1(class_t1, class_t2, watershed_mask=None, min_blob_pixels=MIN_BLOB_PIXELS_DEFAULT):
    change = diff_to_change_map(class_t1, class_t2)
    change = geofence_mask(change, watershed_mask)
    return filter_small_blobs(change, min_blob_pixels)

def summarize_changes(change_map, pixel_area_m2=100.0):
    summary = {}
    for cls, name in CHANGE_CLASS_NAMES.items():
        px = int(np.sum(change_map == cls))
        summary[name] = {'pixels': px, 'hectares': round(px * pixel_area_m2 / 10000, 2)}
    return summary


## 7. Rule-based recommendation engine
No ML, fully explainable: combines the class map + change map + a health score into alerts.


In [ ]:
CLASS_HEALTH_WEIGHT = {WATER: 100, DENSE_VEG: 90, AGRI: 70, SPARSE_VEG: 55, FALLOW: 35, BUILTUP: 20, BARREN: 10}

def compute_health_score(class_map):
    # Excludes NODATA_CLASS pixels (no real satellite coverage) -- otherwise
    # a partial-scene gap silently pulls the score toward whatever weight
    # that pixel's spurious model prediction happened to get.
    valid = class_map != NODATA_CLASS
    if not valid.any():
        return 0.0
    weights = np.vectorize(CLASS_HEALTH_WEIGHT.get)(class_map[valid])
    return float(np.mean(weights))

def ndvi_trend(img_t1, img_t2):
    # img_*: full (6,H,W) R,G,B,NIR,NDVI,NDWI stack. Excludes pixels with no
    # real coverage (R,G,B,NIR all exactly 0) in either date -- those read as
    # NDVI=0 (0/0), which biases the trend toward zero if left in.
    no_cov_t1 = np.all(img_t1[:4] == 0, axis=0)
    no_cov_t2 = np.all(img_t2[:4] == 0, axis=0)
    valid = ~(no_cov_t1 | no_cov_t2)
    if not valid.any():
        return 0.0
    return float(np.mean(img_t2[4][valid]) - np.mean(img_t1[4][valid]))

def generate_alerts(class_map_t2, change_map, health_score, ndvi_trend_value,
                     health_history=None, known_project_mask=None, pixel_area_m2=100.0):
    alerts = []
    def area_ha(mask):
        return round(int(np.sum(mask)) * pixel_area_m2 / 10000, 2)
    construction_mask = change_map == 2
    if construction_mask.any():
        verified = known_project_mask if known_project_mask is not None else np.zeros_like(construction_mask)
        unverified = construction_mask & ~verified
        if unverified.any():
            alerts.append({'severity': 'ALERT', 'rule': 'new_construction',
                'message': 'Possible unauthorized construction detected -- recommend field verification.',
                'area_ha': area_ha(unverified)})
    degradation_mask = change_map == 3
    if degradation_mask.any() and ndvi_trend_value < -0.02:
        alerts.append({'severity': 'RECOMMEND', 'rule': 'degradation_intervention',
            'message': 'Vegetation/water loss with declining NDVI trend -- soil/water conservation '
                       'structure recommended in this zone.',
            'area_ha': area_ha(degradation_mask)})
    new_water_mask = change_map == 1
    if new_water_mask.any():
        matched = known_project_mask if known_project_mask is not None else np.zeros_like(new_water_mask)
        confirmed = new_water_mask & matched
        if confirmed.any():
            alerts.append({'severity': 'VERIFIED', 'rule': 'new_structure_confirmed',
                'message': 'New conservation structure confirmed within a known project -- update project records.',
                'area_ha': area_ha(confirmed)})
        unmatched = new_water_mask & ~matched
        if unmatched.any():
            alerts.append({'severity': 'INFO', 'rule': 'new_water_unverified',
                'message': 'New water body detected outside known project boundaries -- worth a field check.',
                'area_ha': area_ha(unmatched)})
    history = list(health_history or []) + [health_score]
    if len(history) >= 2 and all(h < 40 for h in history[-2:]):
        alerts.append({'severity': 'RECOMMEND', 'rule': 'priority_intervention',
            'message': f'Watershed health declining for {len(history)}+ consecutive periods '
                       f'(current score={health_score:.1f}/100) -- priority intervention recommended.',
            'area_ha': None})
    if not alerts:
        alerts.append({'severity': 'INFO', 'rule': 'no_flags',
            'message': 'No alerts this period -- watershed conditions stable.', 'area_ha': None})
    return alerts


## 8. Model 2 — Siamese change-detection U-Net (optional, build after Model 1 works)
Shared ResNet18 encoder (warm-started from Model 1) processes T1/T2 independently; `|difference|` of encoder features feeds a U-Net decoder to a 5-class change map. Fine-tuned on **self-generated weak labels** from the Tier-1 diff above -- no manual annotation.


In [ ]:
import torch.nn as nn
NUM_CHANGE_CLASSES = 5

class SiameseChangeUNet(nn.Module):
    def __init__(self, encoder_name='resnet18', in_channels=IN_CHANNELS,
                 num_classes=NUM_CHANGE_CLASSES, pretrained_encoder_ckpt=None):
        super().__init__()
        base = smp.Unet(encoder_name=encoder_name, encoder_weights='imagenet',
                         in_channels=in_channels, classes=num_classes)
        self.encoder = base.encoder
        self.decoder = base.decoder
        self.head = base.segmentation_head
        if pretrained_encoder_ckpt is not None and Path(pretrained_encoder_ckpt).exists():
            ckpt = torch.load(pretrained_encoder_ckpt, map_location='cpu')
            state = {k.replace('encoder.', ''): v for k, v in ckpt['model_state'].items() if k.startswith('encoder.')}
            result = self.encoder.load_state_dict(state, strict=False)
            if result is not None:
                missing, unexpected = result
                print(f'Warm-started encoder from {Path(pretrained_encoder_ckpt).name} '
                      f'(missing={len(missing)}, unexpected={len(unexpected)})')
            else:
                # smp's EncoderMixin.load_state_dict doesn't return (missing, unexpected)
                # like a normal nn.Module -- the load still happens, just silently.
                print(f'Warm-started encoder from {Path(pretrained_encoder_ckpt).name}')
    def forward(self, img_t1, img_t2):
        feats_t1 = self.encoder(img_t1)
        feats_t2 = self.encoder(img_t2)
        diff_feats = [torch.abs(a - b) for a, b in zip(feats_t1, feats_t2)]
        decoded = self.decoder(diff_feats)
        return self.head(decoded)

class WeakLabelChangeDataset(Dataset):
    def __init__(self, tiles_dir, aoi_name):
        t1_files = sorted(tiles_dir.glob(f'{aoi_name}_T1_p*_a0.npz'))
        self.pairs = []
        for t1_path in t1_files:
            t2_path = tiles_dir / t1_path.name.replace('_T1_', '_T2_')
            if t2_path.exists():
                self.pairs.append((t1_path, t2_path))
        if not self.pairs:
            raise RuntimeError('No matching T1/T2 tile pairs -- run tiling cell first.')
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        t1_path, t2_path = self.pairs[idx]
        d1, d2 = np.load(t1_path), np.load(t2_path)
        change_label = diff_to_change_map(d1['mask'], d2['mask'])
        return (torch.from_numpy(d1['image'].astype('float32')),
                torch.from_numpy(d2['image'].astype('float32')),
                torch.from_numpy(change_label.astype('int64')))

def compute_change_class_weights(dataset, num_classes):
    # Median-frequency balancing over the weak change labels -- same fix, same
    # reasoning as compute_class_weights above for Model 1 (Fallow collapsed to
    # IoU 0.000 under an unweighted loss + val_loss-only checkpoint selection;
    # 'No change' dominates a weak-label change map just as heavily, so the real
    # change classes are the ones at risk here).
    counts = np.zeros(num_classes, dtype='int64')
    for i in range(len(dataset)):
        _, _, label = dataset[i]
        counts += np.bincount(label.numpy().ravel(), minlength=num_classes)
    freq = counts / counts.sum()
    present = freq > 0
    median_freq = np.median(freq[present])
    weights = np.where(present, median_freq / np.maximum(freq, 1e-12), 0.0)
    print('Change-class weights (median-frequency balanced):')
    for c in range(num_classes):
        print(f'  {CHANGE_CLASS_NAMES.get(c, c):<38} count={counts[c]:>10}  '
              f'freq={freq[c]*100:6.2f}%  weight={weights[c]:6.3f}')
    return torch.tensor(weights, dtype=torch.float32)


In [ ]:
weak_ds = WeakLabelChangeDataset(TILES_DIR, AOI_NAME)
n_val2 = max(1, int(len(weak_ds) * 0.15))
train_ds2, val_ds2 = torch.utils.data.random_split(
    weak_ds, [len(weak_ds) - n_val2, n_val2], generator=torch.Generator().manual_seed(42))
train_loader2 = DataLoader(train_ds2, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader2 = DataLoader(val_ds2, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Weak-label pairs: {len(weak_ds)}  (train={len(train_ds2)}, val={len(val_ds2)})')

model2 = SiameseChangeUNet(pretrained_encoder_ckpt=MODELS_DIR / 'model1_lulc_unet.pt').to(device)
change_class_weights = compute_change_class_weights(train_ds2, NUM_CHANGE_CLASSES).to(device)
dice2 = smp.losses.DiceLoss(mode='multiclass')
ce2 = torch.nn.CrossEntropyLoss(weight=change_class_weights)
loss_fn2 = lambda logits, target: dice2(logits, target) + ce2(logits, target)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=LR)
scaler2 = torch.amp.GradScaler('cuda', enabled=(device.type=='cuda'))

# Checkpoint selection by mean IoU, not aggregate val_loss -- same fix as Model 1
# (a rare class can collapse to IoU 0.000 while still 'winning' on val_loss).
best_mean_iou2 = -1.0
ckpt_path2 = MODELS_DIR / 'model2_change_siamese.pt'

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    model2.train()
    train_loss = 0.0
    for img_t1, img_t2, labels in train_loader2:
        img_t1, img_t2, labels = img_t1.to(device), img_t2.to(device), labels.to(device)
        with torch.autocast(device_type=device.type, enabled=(device.type=='cuda')):
            logits = model2(img_t1, img_t2)
            loss = loss_fn2(logits, labels)
        optimizer2.zero_grad(set_to_none=True)
        scaler2.scale(loss).backward()
        scaler2.step(optimizer2)
        scaler2.update()
        train_loss += loss.item()
    train_loss /= max(len(train_loader2), 1)
    model2.eval()
    val_loss = 0.0
    val_cm2 = np.zeros((NUM_CHANGE_CLASSES, NUM_CHANGE_CLASSES), dtype='int64')
    with torch.no_grad():
        for img_t1, img_t2, labels in val_loader2:
            img_t1, img_t2, labels = img_t1.to(device), img_t2.to(device), labels.to(device)
            with torch.autocast(device_type=device.type, enabled=(device.type=='cuda')):
                logits = model2(img_t1, img_t2)
                loss = loss_fn2(logits, labels)
            val_loss += loss.item()
            preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
            val_cm2 += confusion_matrix_from_arrays(labels.cpu().numpy(), preds, NUM_CHANGE_CLASSES)
    val_loss /= max(len(val_loader2), 1)
    mean_iou2 = metrics_from_confusion(val_cm2, CHANGE_CLASS_NAMES)['mean_iou']
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_mean_iou={mean_iou2:.4f}  ({time.time()-t0:.1f}s)')
    if mean_iou2 > best_mean_iou2:
        best_mean_iou2 = mean_iou2
        torch.save({'model_state': model2.state_dict(), 'epoch': epoch, 'val_loss': val_loss, 'val_mean_iou': mean_iou2}, ckpt_path2)
        print(f'  -> saved best checkpoint ({ckpt_path2.name}, mean_iou={mean_iou2:.4f})')
print(f'Done. Best val_mean_iou={best_mean_iou2:.4f}. Checkpoint: {ckpt_path2}')


## 9. Inference + visualization demo
Runs Model 1 on the full T1/T2 rasters, gets the change map (Tier-1 -- swap in Model 2's output the same way once trained), computes health score + alerts, and renders: a static side-by-side figure and an interactive Folium map — the PS's core 'visualization products' ask.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
import folium
from IPython.display import display, Markdown

def pad_to_multiple(arr, multiple=32):
    c, h, w = arr.shape
    pad_h, pad_w = (-h) % multiple, (-w) % multiple
    return np.pad(arr, ((0,0), (0,pad_h), (0,pad_w)), mode='reflect'), (h, w)

def predict_class_map(model, stack_path, device):
    with rasterio.open(stack_path) as src:
        img = src.read()
        profile = src.profile
    padded, (orig_h, orig_w) = pad_to_multiple(img)
    tensor = torch.from_numpy(padded).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad(), torch.autocast(device_type=device.type, enabled=(device.type=='cuda')):
        logits = model(tensor)
    class_map = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()[:orig_h, :orig_w].astype('uint8')
    # R,G,B,NIR all exactly 0 = no real satellite coverage (partial scene footprint),
    # not a real reading -- override with the nodata sentinel instead of letting the
    # model assign it a plausible-looking wrong class (e.g. a solid fake 'water' blob).
    nodata = np.all(img[:4] == 0, axis=0)
    class_map[nodata] = NODATA_CLASS
    return class_map, img, profile

def render_lulc_map(class_map, ax, title):
    cmap = ListedColormap([np.array(CLASS_COLORS[i]) / 255 for i in range(NUM_CLASSES)])
    cmap.set_over(np.array(CLASS_COLORS[NODATA_CLASS]) / 255)
    norm = Normalize(vmin=0, vmax=NUM_CLASSES-1, clip=False)
    ax.imshow(class_map, cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(title); ax.axis('off')

def render_change_map(change_map, ax, title):
    change_colors = {0:(230,230,230), 1:(66,135,245), 2:(200,30,30), 3:(139,69,19), 4:(34,139,34)}
    cmap = ListedColormap([np.array(change_colors[i]) / 255 for i in range(5)])
    ax.imshow(change_map, cmap=cmap, vmin=0, vmax=4, interpolation='nearest')
    ax.set_title(title); ax.axis('off')

def build_folium_map(class_map, profile, out_path):
    height, width = class_map.shape
    color_img = np.zeros((height, width, 3), dtype='uint8')
    for cls, rgb in CLASS_COLORS.items():
        color_img[class_map == cls] = rgb
    bounds = rasterio.transform.array_bounds(height, width, profile['transform'])
    minx, miny, maxx, maxy = transform_bounds(profile['crs'], 'EPSG:4326', *bounds)
    center = [(miny+maxy)/2, (minx+maxx)/2]
    fmap = folium.Map(location=center, zoom_start=14, tiles='OpenStreetMap')
    folium.raster_layers.ImageOverlay(image=color_img, bounds=[[miny,minx],[maxy,maxx]],
                                       opacity=0.65, name='LULC (Model 1)').add_to(fmap)
    folium.LayerControl().add_to(fmap)
    fmap.save(str(out_path))
    print(f'Saved interactive map: {out_path}')
    return fmap

def render_results_summary(aoi_name, health, trend, change_map, alerts):
    change_stats = summarize_changes(change_map)
    trend_word = 'improving' if trend > 0.01 else ('declining' if trend < -0.01 else 'stable')
    lines = [
        f'## Watershed Report -- {aoi_name}',
        f'**Health score: {health:.0f}/100**  |  **Vegetation trend: {trend_word}** (NDVI {trend:+.4f})',
        '',
        '| Change type | Area |',
        '|---|---|',
    ]
    for name, stats in change_stats.items():
        if stats['hectares'] > 0:
            lines.append(f"| {name} | {stats['hectares']} ha |")
    lines.append('')
    lines.append('**Alerts & recommendations:**')
    icon = {'ALERT': '\U0001F534', 'RECOMMEND': '\U0001F7E1', 'VERIFIED': '\U0001F7E2', 'INFO': '\U0001F535'}
    for a in alerts:
        area = f" ({a['area_ha']} ha)" if a['area_ha'] else ''
        lines.append(f"- {icon.get(a['severity'], '')} **[{a['severity']}]** {a['message']}{area}")
    return '\n'.join(lines)


In [ ]:
ckpt = torch.load(MODELS_DIR / 'model1_lulc_unet.pt', map_location=device)
model1.load_state_dict(ckpt['model_state'])
print(f"Loaded Model 1 (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")

class_t1, img_t1, profile = predict_class_map(model1, DATA_PROCESSED / f'{AOI_NAME}_T1_stack6.tif', device)
class_t2, img_t2, _ = predict_class_map(model1, DATA_PROCESSED / f'{AOI_NAME}_T2_stack6.tif', device)

change_map = run_tier1(class_t1, class_t2)
health = compute_health_score(class_t2)
trend = ndvi_trend(img_t1, img_t2)

alerts = generate_alerts(class_t2, change_map, health, trend)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
render_lulc_map(class_t1, axes[0], 'LULC -- T1')
render_lulc_map(class_t2, axes[1], 'LULC -- T2')
render_change_map(change_map, axes[2], 'Change map (Tier-1)')
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / 'lulc_demo.png', dpi=150)
plt.show()

display(Markdown(render_results_summary(AOI_NAME, health, trend, change_map, alerts)))

fmap = build_folium_map(class_t2, profile, OUTPUTS_DIR / 'watershed_map.html')
display(fmap)  # renders inline in the notebook, not just saved to a file


## 10. Live demo — fast path (use this when presenting to judges)

Everything above (data pull, training) only needs to run **once**, ahead of time. On demo day, don't re-run those cells live — retraining takes minutes and risks a flaky download mid-presentation. Instead: run **GPU check -> Install -> Mount Drive -> Config** at the top of this notebook (fast, ~1 min total), then jump straight to this one self-contained cell. It loads the already-trained checkpoint from Drive and produces the full result in seconds.


In [ ]:
import torch, rasterio, numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from rasterio.warp import transform_bounds
import folium
from IPython.display import display, Markdown
import segmentation_models_pytorch as smp
from scipy import ndimage

assert (MODELS_DIR / 'model1_lulc_unet.pt').exists(), 'No trained checkpoint in Drive yet -- run the training cells at least once first.'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model1 = smp.Unet(encoder_name='resnet18', encoder_weights='imagenet', in_channels=IN_CHANNELS, classes=NUM_CLASSES).to(device)
ckpt = torch.load(MODELS_DIR / 'model1_lulc_unet.pt', map_location=device)
model1.load_state_dict(ckpt['model_state'])
model1.eval()
print(f"Loaded Model 1 (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")

WATER, DENSE_VEG, AGRI, SPARSE_VEG, BARREN, BUILTUP, FALLOW = range(7)

def pad_to_multiple(arr, multiple=32):
    c, h, w = arr.shape
    pad_h, pad_w = (-h) % multiple, (-w) % multiple
    return np.pad(arr, ((0,0), (0,pad_h), (0,pad_w)), mode='reflect'), (h, w)

def predict_class_map(stack_path):
    with rasterio.open(stack_path) as src:
        img = src.read(); profile = src.profile
    padded, (orig_h, orig_w) = pad_to_multiple(img)
    tensor = torch.from_numpy(padded).unsqueeze(0).to(device)
    with torch.no_grad(), torch.autocast(device_type=device.type, enabled=(device.type=='cuda')):
        logits = model1(tensor)
    class_map = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()[:orig_h,:orig_w].astype('uint8')
    # R,G,B,NIR all exactly 0 = no real satellite coverage (partial scene footprint) --
    # override with the nodata sentinel instead of a spurious real-class prediction.
    class_map[np.all(img[:4] == 0, axis=0)] = NODATA_CLASS
    return class_map, img, profile

def diff_to_change_map(t1, t2):
    change = np.zeros(t1.shape, dtype='uint8')
    change[np.isin(t1, [BARREN, SPARSE_VEG, AGRI, FALLOW]) & (t2 == WATER)] = 1
    change[np.isin(t1, [BARREN, SPARSE_VEG, AGRI, FALLOW, WATER]) & (t2 == BUILTUP)] = 2
    change[np.isin(t1, [DENSE_VEG, WATER]) & (t2 == BARREN)] = 3
    change[np.isin(t1, [BARREN, FALLOW, SPARSE_VEG]) & np.isin(t2, [DENSE_VEG, AGRI])] = 4
    return change

def filter_small_blobs(change_map, min_pixels=12):
    cleaned = change_map.copy()
    for cls in np.unique(change_map):
        if cls == 0: continue
        cls_mask = change_map == cls
        labeled, n = ndimage.label(cls_mask)
        sizes = ndimage.sum(cls_mask, labeled, range(1, n+1))
        for blob_id, size in enumerate(sizes, start=1):
            if size < min_pixels: cleaned[labeled == blob_id] = 0
    return cleaned

def summarize_changes(change_map, pixel_area_m2=100.0):
    out = {}
    for cls, name in CHANGE_CLASS_NAMES.items():
        px = int(np.sum(change_map == cls))
        out[name] = {'pixels': px, 'hectares': round(px * pixel_area_m2 / 10000, 2)}
    return out

CLASS_HEALTH_WEIGHT = {WATER:100, DENSE_VEG:90, AGRI:70, SPARSE_VEG:55, FALLOW:35, BUILTUP:20, BARREN:10}
def compute_health_score(class_map):
    valid = class_map != NODATA_CLASS
    if not valid.any(): return 0.0
    return float(np.mean(np.vectorize(CLASS_HEALTH_WEIGHT.get)(class_map[valid])))

def generate_alerts(change_map, health_score, ndvi_trend_value):
    alerts = []
    def area_ha(mask): return round(int(np.sum(mask)) * 100.0 / 10000, 2)
    construction_mask = change_map == 2
    if construction_mask.any():
        alerts.append({'severity': 'ALERT', 'message': 'Possible unauthorized construction detected -- recommend field verification.', 'area_ha': area_ha(construction_mask)})
    degradation_mask = change_map == 3
    if degradation_mask.any() and ndvi_trend_value < -0.02:
        alerts.append({'severity': 'RECOMMEND', 'message': 'Vegetation/water loss with declining NDVI trend -- soil/water conservation structure recommended.', 'area_ha': area_ha(degradation_mask)})
    new_water_mask = change_map == 1
    if new_water_mask.any():
        alerts.append({'severity': 'INFO', 'message': 'New water body detected -- worth a field check.', 'area_ha': area_ha(new_water_mask)})
    if not alerts:
        alerts.append({'severity': 'INFO', 'message': 'No alerts this period -- watershed conditions stable.', 'area_ha': None})
    return alerts

def render_lulc_map(class_map, ax, title):
    cmap = ListedColormap([np.array(CLASS_COLORS[i]) / 255 for i in range(NUM_CLASSES)])
    cmap.set_over(np.array(CLASS_COLORS[NODATA_CLASS]) / 255)
    norm = Normalize(vmin=0, vmax=NUM_CLASSES-1, clip=False)
    ax.imshow(class_map, cmap=cmap, norm=norm, interpolation='nearest')
    ax.set_title(title); ax.axis('off')

def render_change_map(change_map, ax, title):
    change_colors = {0:(230,230,230), 1:(66,135,245), 2:(200,30,30), 3:(139,69,19), 4:(34,139,34)}
    cmap = ListedColormap([np.array(change_colors[i]) / 255 for i in range(5)])
    ax.imshow(change_map, cmap=cmap, vmin=0, vmax=4, interpolation='nearest')
    ax.set_title(title); ax.axis('off')

def render_results_summary(aoi_name, health, trend, change_map, alerts):
    change_stats = summarize_changes(change_map)
    trend_word = 'improving' if trend > 0.01 else ('declining' if trend < -0.01 else 'stable')
    lines = [f'## Watershed Report -- {aoi_name}',
             f'**Health score: {health:.0f}/100**  |  **Vegetation trend: {trend_word}** (NDVI {trend:+.4f})',
             '', '| Change type | Area |', '|---|---|']
    for name, stats in change_stats.items():
        if stats['hectares'] > 0: lines.append(f"| {name} | {stats['hectares']} ha |")
    lines.append(''); lines.append('**Alerts & recommendations:**')
    icon = {'ALERT': '\U0001F534', 'RECOMMEND': '\U0001F7E1', 'VERIFIED': '\U0001F7E2', 'INFO': '\U0001F535'}
    for a in alerts:
        area = f" ({a['area_ha']} ha)" if a['area_ha'] else ''
        lines.append(f"- {icon.get(a['severity'], '')} **[{a['severity']}]** {a['message']}{area}")
    return '\n'.join(lines)

# --- run it ---
class_t1, img_t1, profile = predict_class_map(DATA_PROCESSED / f'{AOI_NAME}_T1_stack6.tif')
class_t2, img_t2, _ = predict_class_map(DATA_PROCESSED / f'{AOI_NAME}_T2_stack6.tif')
change_map = filter_small_blobs(diff_to_change_map(class_t1, class_t2))
health = compute_health_score(class_t2)
# Excludes pixels with no real coverage (R,G,B,NIR all exactly 0) in either date --
# those read as NDVI=0 (0/0), which would bias the trend toward zero if left in.
_no_cov = np.all(img_t1[:4] == 0, axis=0) | np.all(img_t2[:4] == 0, axis=0)
_valid = ~_no_cov
trend = float(np.mean(img_t2[4][_valid]) - np.mean(img_t1[4][_valid])) if _valid.any() else 0.0
alerts = generate_alerts(change_map, health, trend)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
render_lulc_map(class_t1, axes[0], 'LULC -- T1')
render_lulc_map(class_t2, axes[1], 'LULC -- T2')
render_change_map(change_map, axes[2], 'Change map (Tier-1)')
fig.tight_layout(); plt.show()

display(Markdown(render_results_summary(AOI_NAME, health, trend, change_map, alerts)))

height, width = class_t2.shape
color_img = np.zeros((height, width, 3), dtype='uint8')
for cls, rgb in CLASS_COLORS.items(): color_img[class_t2 == cls] = rgb
bounds = rasterio.transform.array_bounds(height, width, profile['transform'])
minx, miny, maxx, maxy = transform_bounds(profile['crs'], 'EPSG:4326', *bounds)
fmap = folium.Map(location=[(miny+maxy)/2, (minx+maxx)/2], zoom_start=14, tiles='OpenStreetMap')
folium.raster_layers.ImageOverlay(image=color_img, bounds=[[miny,minx],[maxy,maxx]], opacity=0.65, name='LULC').add_to(fmap)
folium.LayerControl().add_to(fmap)
display(fmap)
